In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder

# Define your emotion labels
emotion_labels = ['ANGRY', 'DISGUST', 'FEAR', 'HAPPY', 'SAD', 'SURPRISE']  # Adjust based on your training labels

# Initialize and fit the LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(emotion_labels)

# Load your trained emotion detection model
model = load_model('emotion_detection_cnn_model.h5')

# Load the pre-trained face detector from OpenCV
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


In [ ]:
# Start capturing video from the webcam
cap = cv2.VideoCapture(0)

while True:
    # Read a frame from the video capture
    ret, frame = cap.read()
    if not ret:
        break

    # Convert the frame to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detect faces in the frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Process each detected face
    for (x, y, w, h) in faces:
        # Extract the region of interest (the face)
        roi_gray = gray[y:y+h, x:x+w]
        
        # Preprocess the face image
        roi_resized = cv2.resize(roi_gray, (48, 48))
        roi_normalized = roi_resized / 255.0
        roi_reshaped = roi_normalized.reshape(1, 48, 48, 1)  # Add channel dimension
        
        # Make a prediction using the model
        prediction = model.predict(roi_reshaped)
        emotion = label_encoder.inverse_transform([np.argmax(prediction)])[0]
        
        # Draw a rectangle around the face and put the predicted emotion text
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, emotion, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
    
    # Display the frame with the detected faces and predicted emotions
    cv2.imshow('Real-time Face Emotion Detector', frame)
    
    # Break the loop if the 'q' key is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and close the display window
cap.release()
cv2.destroyAllWindows()
